# TheraBot with LoRA Training

This notebook implements fine-tuning of Llama-3.2-1B-Instruct for mental health counseling using LoRA (Low-Rank Adaptation). The model is trained on mental health counseling conversations to provide empathetic, professional guidance.

## Overview
- **Base Model**: meta-llama/Llama-3.2-1B-Instruct
- **Training Method**: Supervised Fine-Tuning with LoRA
- **Dataset**: Mental health counseling conversations
- **Objective**: Create a compassionate AI counselor that provides supportive guidance

## Prerequisites
- Access to Hugging Face models (login required)
- GPU runtime recommended for training
- MLflow for experiment tracking

## 1. Environment Setup

### Install Required Packages

In [ ]:
!pip install -q mlflow
!pip install -q --upgrade transformers datasets accelerate peft evaluate
!pip install -q huggingface_hub

### Import Libraries and Configuration

In [ ]:
# Import required libraries
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    DataCollatorForLanguageModeling, TrainingArguments, Trainer
)
import torch
import math
import mlflow
import os

# Training Configuration
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
OUTPUT_DIR = "./outputs-llama1b-counselor"
EXPERIMENT_NAME = "mental-health-chatbot"
RUN_NAME = "llama1b_counselor_sft"

# Training Hyperparameters
MAX_LEN = 2048          # Maximum sequence length
EPOCHS = 3              # Number of training epochs
LR = 1e-5              # Learning rate
BSZ = 4                # Batch size per device
GRAD_ACCUM = 4         # Gradient accumulation steps

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

### Hugging Face Authentication

In [ ]:
from huggingface_hub import login
login()

## 2. Data Preparation

### Load Dataset

In [ ]:
# Load the mental health counseling dataset
dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")
print(f"Dataset size: {len(dataset)}")
print("Columns:", dataset.column_names)
print("Sample row:", dataset[0])

# Split dataset into train and validation sets
ds = dataset.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = ds["train"], ds["test"]
print(f"Train/Eval sizes: {len(train_ds)} / {len(eval_ds)}")

### Format Data for Chat Template

In [ ]:
# System prompt for the counselor
system_prompt = """You are a professional counselor and mental health advisor. Your role is to:

- Listen empathetically and provide supportive, evidence-based guidance
- Ask clarifying questions when needed to better understand the situation
- Offer practical coping strategies and actionable advice
- Recognize when professional help may be needed and provide appropriate referrals
- Maintain a warm, non-judgmental tone while being direct and helpful
- Focus on empowering the person to develop healthy coping mechanisms

Always prioritize the person's safety and well-being in your responses."""

# Llama chat template tokens
BOS = "<|begin_of_text|>"
EOT = "<|eot_id|>"
EOS = "<|end_of_text|>"

def _role(r):
    """Helper function to format role headers"""
    return f"<|start_header_id|>{r}<|end_header_id|>\n\n"

def fmt(ex):
    """Format a single example into Llama chat format"""
    context = str(ex["Context"]).strip()
    response = str(ex["Response"]).strip()

    # Enhance the user context with clearer instruction
    enhanced_context = (
        "A person is seeking guidance with the following concern:\n\n"
        f"{context}\n\n"
        "Please provide thoughtful, professional advice."
    )

    # Build the conversation in Llama format
    text = (
        f"{BOS}"
        f"{_role('system')}{system_prompt}{EOT}"
        f"{_role('user')}{enhanced_context}{EOT}"
        f"{_role('assistant')}{response}{EOT}{EOS}"
    )
    return {"text": text}

# Apply formatting to both datasets
formatted_train = train_ds.map(fmt, remove_columns=train_ds.column_names)
formatted_eval = eval_ds.map(fmt, remove_columns=eval_ds.column_names)

print("Formatted example:\n", formatted_train[0]["text"][:400])

## 3. Model & Tokenizer Setup

### Load Model and Tokenizer

### Prepare Labels for Assistant-Only Training

This section masks tokens to only compute loss on the assistant responses, not the user prompts.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tok_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

tokenized_train = formatted_train.map(tok_fn, batched=True, remove_columns=["text"])
#tokenized_eval  = formatted_eval.map(tok_fn,  batched=True, remove_columns=["text"])

print("Tokenized keys:", tokenized_train.features)

In [ ]:
ASSISTANT_HDR = "<|start_header_id|>assistant<|end_header_id|>\n\n"
ah_ids  = tokenizer(ASSISTANT_HDR, add_special_tokens=False)["input_ids"]
eot_ids = tokenizer(EOT,           add_special_tokens=False)["input_ids"]

def _find_subseq(h, n):
    L, N = len(h), len(n)
    for i in range(L - N + 1):
        if h[i:i+N] == n:
            return i
    return None

def tok_fn_assistant_only(batch):
    out = tokenizer(batch["text"], padding="max_length", truncation=True, max_length=MAX_LEN)
    labels = []
    for ids in out["input_ids"]:
        lab = ids.copy()
        start = _find_subseq(ids, ah_ids)
        if start is None:
            lab[:] = [-100] * len(lab)
        else:
            s = start + len(ah_ids)
            rel_end = _find_subseq(ids[s:], eot_ids)
            e = s + rel_end if rel_end is not None else len(ids)
            for i in range(0, s):        lab[i] = -100
            for i in range(e, len(lab)): lab[i] = -100
        labels.append(lab)
    out["labels"] = labels
    return out

tokenized_eval = formatted_eval.map(tok_fn_assistant_only, batched=True, remove_columns=["text"])
print("Assistant-only eval prepared:", len(tokenized_eval), "examples")

## 4. Model Training

### Initialize Model and Data Collator

In [ ]:
# %% ------------------ Model ------------------
dtype = (
    torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else (torch.float16 if torch.cuda.is_available() else torch.float32)
)
print("Using dtype:", dtype)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32, #dtype,
    device_map="auto",
    trust_remote_code=True,
)

model.config.pad_token_id = tokenizer.pad_token_id

# %% ------------------ Collator ------------------
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

### Configure Training Arguments & Start Training

Training with MLflow integration for experiment tracking.

In [ ]:
# %% ------------------ Training Args (MLflow integrated) ------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BSZ,
    per_device_eval_batch_size=BSZ,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=0.0,

    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    save_total_limit=2,

    fp16=False, #(dtype==torch.float16),
    bf16=True, #(dtype==torch.bfloat16),
    gradient_checkpointing=True,
    report_to=["mlflow"],             # <-- key line for Transformers->MLflow logging
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=collator,
)

# %% ------------------ Train + Evaluate with MLflow ------------------
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME):
    # Helpful params in the run
    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("max_len", MAX_LEN)
    mlflow.log_param("lr", LR)
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("per_device_bsz", BSZ)
    mlflow.log_param("grad_accum", GRAD_ACCUM)
    mlflow.log_param("dtype", str(dtype))

    print("Starting training...")
    train_result = trainer.train()
    trainer.log_metrics("train", train_result.metrics)
    trainer.save_metrics("train", train_result.metrics)

    print("Evaluating...")
    eval_metrics = trainer.evaluate()
    # Add perplexity for convenience
    if "eval_loss" in eval_metrics and eval_metrics["eval_loss"] is not None:
        eval_metrics["eval_perplexity"] = float(math.exp(eval_metrics["eval_loss"]))
    print("Eval metrics:", eval_metrics)

    # Log and save
    trainer.log_metrics("eval", eval_metrics)
    trainer.save_metrics("eval", eval_metrics)
    trainer.save_state()
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

print(" Done. MLflow runs saved under ./mlruns. Launch UI with: mlflow ui --port 5000")

## 5. Model Export

### Package Model for Inference

Creates a lightweight zip file with only the essential files needed for inference.

In [ ]:
# Pack only inference essentials from your save dir, then download the zip.
import os, zipfile, pathlib
#from google.colab import files

MODEL_DIR = "/content/outputs-llama1b-counselor"   # <- change if yours is different
ARCHIVE   = "/content/counselor_inference_only.zip"

# Always-try list (added only if present)
KEEP = {
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "generation_config.json",     # optional but handy
    "chat_template.jinja",        # optional if you want apply_chat_template
    "tokenizer.model",            # SentencePiece (some models)
    "spiece.model",               # SentencePiece (older name)
    "merges.txt", "vocab.json",   # BPE (some tokenizers)
    "added_tokens.json",          # if you added any special tokens
}

# Also include any safetensors shards (handles single or sharded weights)
def list_weight_files(root):
    out = []
    for f in os.listdir(root):
        if f.endswith(".safetensors") or f.endswith(".safetensors.index.json"):
            out.append(f)
    return sorted(out)

weight_files = list_weight_files(MODEL_DIR)
if not weight_files:
    raise FileNotFoundError("No *.safetensors weights found in MODEL_DIR")

with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_DEFLATED) as z:
    # weights
    for f in weight_files:
        z.write(os.path.join(MODEL_DIR, f),
                arcname=os.path.join("outputs-llama1b-counselor", f))
    # configs/tokenizer/etc.
    for f in KEEP:
        p = os.path.join(MODEL_DIR, f)
        if os.path.exists(p):
            z.write(p, arcname=os.path.join("outputs-llama1b-counselor", f))

print("Wrote:", ARCHIVE)
#files.download(ARCHIVE)

## Structured Pruning

In [ ]:
# --- Structured Pruning Step ---
import torch.nn.utils.prune as prune

def apply_structured_pruning(model, amount=0.2):
    """
    Apply structured pruning on Linear layers in the model.
    amount = fraction of neurons to prune (e.g., 0.2 = prune 20%)
    """
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            # Prune entire output channels (structured pruning)
            prune.ln_structured(module, name="weight", amount=amount, n=2, dim=0)
            prune.remove(module, "weight")  # Make pruning permanent
    return model

# Start MLflow run for pruning
with mlflow.start_run(run_name="model_pruning"):
    # Log pruning parameters
    mlflow.log_param("pruning_method", "structured")
    mlflow.log_param("pruning_amount", 0.2)
    mlflow.log_param("target_layers", "Linear")
    
    # Calculate model size before pruning
    model_size_before = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**3
    mlflow.log_metric("model_size_before_gb", model_size_before)
    
    print(" Applying structured pruning (20%)...")
    model = apply_structured_pruning(model, amount=0.2)
    
    # Calculate model size after pruning
    model_size_after = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**3
    compression_ratio = model_size_before / model_size_after
    
    mlflow.log_metric("model_size_after_gb", model_size_after)
    mlflow.log_metric("compression_ratio", compression_ratio)
    mlflow.log_metric("size_reduction_percent", (1 - model_size_after/model_size_before) * 100)

# Save pruned model before quantization
PRUNED_DIR = "./onnx_model_pruned"
os.makedirs(PRUNED_DIR, exist_ok=True)
model.save_pretrained(PRUNED_DIR)
tokenizer.save_pretrained(PRUNED_DIR)

print(f"✅ Structured pruning complete. Pruned model saved at {PRUNED_DIR}")

## Quantization

In [ ]:
# Install llama.cpp for GGUF conversion
!git clone https://github.com/ggerganov/llama.cpp.git
!sudo apt-get install -y cmake
!cd llama.cpp && cmake -j

# Install Python requirements for conversion
!cd llama.cpp && pip install -r requirements.txt

In [ ]:
# Test loading the model
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

#print(f"Loading model from: {MODEL_PATH}")

# Load tokenizer and model
model = AutoModelForCausalLM.from_pretrained("models/fp32", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("models/fp32")

print(f" Model loaded successfully!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Model size: {sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**3:.2f} GB")

In [ ]:
%cd /content/llama.cpp
!mkdir -p build && cd build && cmake .. && cmake --build . --config Release

In [ ]:
# Start MLflow run for quantization
with mlflow.start_run(run_name="model_quantization"):
    # Log quantization parameters
    mlflow.log_param("quantization_method", "GGUF_Q8_0")
    mlflow.log_param("input_format", "fp16")
    mlflow.log_param("output_format", "int8")
    
    # Get file sizes for comparison
    import os
    if os.path.exists("/content/therabot_fp16.gguf"):
        fp16_size = os.path.getsize("/content/therabot_fp16.gguf") / 1024**3
        mlflow.log_metric("fp16_model_size_gb", fp16_size)
    
    !./build/bin/llama-quantize /content/therabot_fp16.gguf /content/therabot_int8.gguf Q8_0
    
    # Log quantized model size
    if os.path.exists("/content/therabot_int8.gguf"):
        int8_size = os.path.getsize("/content/therabot_int8.gguf") / 1024**3
        mlflow.log_metric("int8_model_size_gb", int8_size)
        
        if 'fp16_size' in locals():
            compression_ratio = fp16_size / int8_size
            mlflow.log_metric("quantization_compression_ratio", compression_ratio)
            mlflow.log_metric("quantization_size_reduction_percent", (1 - int8_size/fp16_size) * 100)